# 01 生データの目視確認

実車で記録した最初の 9 本（`data/raw/ride_001`〜`ride_009`）を読み、列・周波数・値域・時刻・GPS の有効割合を確認する。
結論は `docs/data_schema.md` にまとめた。このノートブックは確認の記録で、処理の本体は `src/ct125/` にある。

**位置情報の扱い**：緯度・経度の実値は表示しない（記録した地域の箱に入るかの判定と、最初の fix からの相対距離だけを出す）。

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Meiryo", "Yu Gothic", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
RAW = Path("../data/raw")
# 記録した地域のおおまかな箱（判定にだけ使う）。値は config/local.yaml（コミットしない）の area_bbox。無ければ判定を飛ばす
import yaml
_local = Path("../config/local.yaml")
AREA_BBOX = (yaml.safe_load(_local.read_text(encoding="utf-8")) or {}).get("area_bbox") if _local.exists() else None

def gps_unix(g):
    d = pd.to_datetime(g["date"].astype(str).str.zfill(6), format="%d%m%y", utc=True)
    u = g["utc"].astype(float)
    sec = (u // 10000) * 3600 + (u // 100 % 100) * 60 + u % 100
    return (d - pd.Timestamp(0, tz="UTC")).dt.total_seconds() + sec

def load(name):
    d = RAW / name
    acc = pd.read_csv(d / "accel.csv")
    gps = pd.read_csv(d / "gps.csv", dtype={"utc": str, "date": str})
    ev = pd.read_csv(d / "events.csv")
    st = pd.read_csv(d / "status.csv")
    meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    memo = " ".join(p.read_text(encoding="utf-8").strip() for p in d.glob("*.txt"))
    acc["t_rel"] = (acc.t_us - acc.t_us.iloc[0]) / 1e6
    acc["mag_g"] = np.sqrt(acc.ax_g**2 + acc.ay_g**2 + acc.az_g**2)
    gps["t_rel"] = (gps.t_rx_us - acc.t_us.iloc[0]) / 1e6
    gv = gps[gps.status == "A"]
    offset = float(np.median(gps_unix(gv) - gv.t_rx_us / 1e6)) if len(gv) else None
    if offset is not None:
        for df, col in ((acc, "t_us"), (gps, "t_rx_us")):
            df["jst"] = pd.to_datetime(df[col] / 1e6 + offset, unit="s", utc=True).dt.tz_convert("Asia/Tokyo").dt.tz_localize(None)
    return dict(name=name, acc=acc, gps=gps, events=ev, status=st, meta=meta, memo=memo, offset=offset)

RIDES = [f"ride_{i:03d}" for i in range(1, 10)]

## 1. 9 本の棚卸し（行数・周波数・欠け・GPS 有効割合）

In [ ]:
rows = []
for name in RIDES:
    r = load(name); acc, gps, st = r["acc"], r["gps"], r["status"]
    dt = np.diff(acc.t_us); dur = (acc.t_us.iloc[-1] - acc.t_us.iloc[0]) / 1e6
    gv = gps[gps.status == "A"]
    first10 = acc[acc.t_rel < 10]
    rows.append(dict(
        ride=name, memo=r["memo"][:28], min=round(dur / 60, 1), acc_rows=len(acc),
        fs_hz=round((len(acc) - 1) / dur, 1), dt_min_us=int(dt.min()), dt_max_us=int(dt.max()),
        n_gaps=int((np.diff(acc.n) != 1).sum()), t_dup=int((dt == 0).sum()), monotonic=bool((dt > 0).all()),
        clip_rows=int((acc[["ax_g", "ay_g", "az_g"]].abs() >= 15.9).any(axis=1).sum()),
        mag_med=round(float(acc.mag_g.median()), 3), mag_p99=round(float(acc.mag_g.quantile(0.99)), 2), mag_max=round(float(acc.mag_g.max()), 1),
        g0=tuple(first10[["ax_g", "ay_g", "az_g"]].mean().round(2)),
        gps_rows=len(gps), gps_fix_ratio=round(len(gv) / len(gps), 3),
        sats=f"{int(gv.sats.min())}-{int(gv.sats.max())}" if len(gv) else "0",
        first_fix_s=round(float(gv.t_rel.iloc[0]), 1) if len(gv) else None,
        jst_start=acc.jst.iloc[0].strftime("%m-%d %H:%M:%S") if "jst" in acc else None,
        in_area=round(float((gv.lat.between(*AREA_BBOX["lat"]) & gv.lon.between(*AREA_BBOX["lon"])).mean()), 2) if len(gv) and AREA_BBOX else None,
        events=",".join(r["events"].type.tolist()[1:]) or "-",
        errors=int(st[["ring_drops", "fifo_overruns", "i2c_errors", "sd_write_errors", "time_resyncs"]].iloc[-1].sum()),
    ))
summary = pd.DataFrame(rows).set_index("ride")
pd.set_option("display.width", 250, "display.max_columns", 50)
summary

- 9 本とも `n` の飛び・`t_us` の重複なし、`events.csv`／`status.csv` のエラー系は 0。実測 fs は 397.5〜400.2 Hz（設定 400 Hz）。
- ride_001〜005 は GPS あり（起動後 4〜38 秒で初回 fix）。ride_006〜009 は先頭から最後まで衛星 0。
- 有効な座標はすべて記録した地域の箱に入る（表 1 の地域の判定が 1.0）。

## 2. 列・型・先頭 5 行（ride_004）

In [ ]:
r4 = load("ride_004")
print(r4["meta"])
display(r4["acc"][["n", "t_us", "ax_g", "ay_g", "az_g"]].head())
print(r4["acc"][["n", "t_us", "ax_g", "ay_g", "az_g"]].dtypes.to_dict())
# gps.csv：先頭は測位前（status=V、lat/lon は空）。lat/lon 列は常にマスクして表示する
g = r4["gps"].drop(columns=["t_rel", "jst"], errors="ignore").copy()
g[["lat", "lon"]] = g[["lat", "lon"]].notna().map(lambda b: "(値あり)" if b else "")
display(g.head())
print(r4["gps"].drop(columns=["t_rel", "jst"], errors="ignore").dtypes.to_dict())
display(r4["events"]); display(r4["status"].head(3))

## 3. 加速度のサンプリング間隔（PLL 方式の確認）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, name in zip(axes, ["ride_004", "ride_007", "ride_009"]):
    acc = load(name)["acc"]; dt = np.diff(acc.t_us)
    ax.hist(dt, bins=np.arange(2450, 2600, 2), color="#4a6fa5")
    ax.set_title(f"{name}: dt 中央値 {np.median(dt):.0f} us → fs {1e6/np.median(dt):.1f} Hz")
    ax.set_xlabel("t_us の間隔 [us]"); ax.set_yscale("log")
plt.tight_layout()

間隔は 2,460〜2,590 us にほぼそろう（古い形（V2）の記録で見られた 2 ms／3 ms 割れはない）。1 秒窓の中では等間隔として扱って問題ない。

## 4. 加速度の値域・重力・振り切れ

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
for ax, name in zip(axes, ["ride_004", "ride_007"]):
    r = load(name); acc = r["acc"]
    w = acc.groupby((acc.t_rel // 1).astype(int))
    rms = w.mag_g.apply(lambda s: np.sqrt(((s - s.mean())**2).mean()))
    mx = w.mag_g.max()
    ax.plot(rms.index, rms.values, lw=0.8, label="1 秒窓 RMS（重力除去後の合成加速度）")
    ax.plot(mx.index, mx.values, lw=0.5, alpha=0.5, label="1 秒窓の最大 合成加速度")
    ax.axhline(15.9, color="r", ls="--", lw=0.8, label="±16g の上限付近")
    ax.set_title(f"{name}（{r['memo'] or 'memo なし'}）"); ax.set_xlabel("走行開始からの秒"); ax.set_ylabel("g"); ax.legend(loc="upper right")
plt.tight_layout()
clip = {n: int((load(n)["acc"][["ax_g", "ay_g", "az_g"]].abs() >= 15.9).sum().sum()) for n in RIDES}
print("|axis| >= 15.9 g のサンプル数:", clip)
a7 = load("ride_007")["acc"]
print("ride_007 で |axis| >= 15 g のサンプル数（軸別）:", (a7[["ax_g", "ay_g", "az_g"]].abs() >= 15).sum().to_dict())

- 単位は g で重力込み。静止時の合成加速度は 0.95〜0.97 g（1 g より数 % 小さいのは ADXL345 のオフセット・感度の個体差。相対的な振動量の評価には影響しない）。
- 振り切れは ride_007 の X 軸で 1 サンプル（15.9 g）、15 g 以上が 25 サンプル。他は 0。郊外走行で X 軸が上限に近づくので、前処理で `is_clipped` を付ける。
- 静止 10 秒の重力の向きは Y 軸が主（ay ≈ +0.85 g、az ≈ −0.4 g）。ride_008/009 は向きが変わっている（表 1 の `g0`）。

## 5. GPS：有効割合、品質、速度（ride_004 / ride_005）

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, name in zip(axes, ["ride_004", "ride_005"]):
    r = load(name); g = r["gps"]; gv = g[g.status == "A"]
    ax.plot(gv.jst, gv.speed_kmh, lw=0.9, label="speed_kmh（RMC）")
    ax2 = ax.twinx(); ax2.plot(gv.jst, gv.sats, color="gray", lw=0.6, label="sats"); ax2.plot(gv.jst, gv.hdop, color="orange", lw=0.6, label="hdop")
    lost = g[g.status == "V"]
    for t in lost.jst: ax.axvspan(t, t + pd.Timedelta(seconds=1), color="red", alpha=0.08, lw=0)
    bad = gv[(gv.sats <= 4) | (gv.hdop > 5)]
    ax.scatter(bad.jst, bad.speed_kmh, color="red", s=10, zorder=3, label="sats<=4 or hdop>5")
    ax.set_title(f"{name}  fix_ratio={len(gv)/len(g):.3f}  開始 {r['acc'].jst.iloc[0]:%H:%M:%S} JST（赤帯＝status V）")
    ax.set_ylabel("km/h"); ax2.set_ylabel("sats / hdop"); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
for name in ["ride_004", "ride_005"]:
    g = load(name)["gps"]
    print(name, "status x fix:")
    print(pd.crosstab(g.status, g.fix.fillna(-1).astype(int)).to_string())

- ride_004 14:16:30〜55 は衛星 3〜6・HDOP 6〜9.7 のまま速度が 29→101 km/h まで単調に増え、直後に `gps_lost`。誤測位として扱い、この区間の速度・座標は使わない。
- ride_005 14:55:23〜24 も衛星 3 で 50〜55 km/h、直後に 61 秒の `gps_lost`。15:02:19〜54 は HDOP 9〜10 が続く。
- `status=A` でも `fix=6`（推定）の行が 1〜2 行ある。有効判定は `status`、品質フィルタは `sats ≥ 5 かつ hdop ≤ 5` で行う（`docs/data_schema.md` 6.）。

## 6. 軌跡の形（最初の fix からの相対距離。緯度経度の実値は出さない）

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
origin = None
for name, c in [("ride_004", "#1f77b4"), ("ride_005", "#d62728")]:
    g = load(name)["gps"]; gv = g[(g.status == "A") & (g.sats >= 5) & (g.hdop <= 5)]
    if origin is None: origin = (gv.lat.iloc[0], gv.lon.iloc[0])
    x = (gv.lon - origin[1]) * 111320 * np.cos(np.deg2rad(origin[0])); y = (gv.lat - origin[0]) * 110540
    ax.plot(x, y, ".", ms=2, color=c, label=f"{name}（品質フィルタ後 {len(gv)} 点）")
ax.set_aspect("equal"); ax.set_xlabel("東西 [m]（ride_004 の最初の fix を 0）"); ax.set_ylabel("南北 [m]"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("ride_004（往路）と ride_005（復路）")
plt.tight_layout()

## 7. GPS なし走行の確認（ride_006〜009）

In [ ]:
for name in ["ride_006", "ride_007", "ride_008", "ride_009"]:
    g = load(name)["gps"]
    print(name, f"rows={len(g)} status_A={int((g.status == 'A').sum())} sats_max={int(g.sats.max())} utc_rows={int(g.utc.notna().sum())} hdop_unique={sorted(g.hdop.unique())}")

4 本とも全行 `status=V`、衛星 0、`utc` なし（時刻すら受信していない）。直前の ride_005 は 33 秒で測位できているので、
コールドスタートの待ち時間ではなく、受信そのものができていない（アンテナの遮蔽、または GPS TX 線の接触）可能性が高い。

## 8. 時刻の同期（t_us と UTC の対応のばらつき）

In [ ]:
for name in RIDES[:5]:
    g = load(name)["gps"]; gv = g[g.status == "A"]
    off = gps_unix(gv) - gv.t_rx_us / 1e6
    print(f"{name}: offset 中央値 {np.median(off):.3f} s, 標準偏差 {off.std()*1e3:.1f} ms, 範囲 {(off.max()-off.min())*1e3:.0f} ms, 有効行 {len(gv)}")

ばらつきは数十 ms（NMEA の送信遅れ）。1 秒窓の解析には十分で、PPS は不要。

## まとめ（`docs/data_schema.md` に反映済み）

| 項目 | 結果 |
|---|---|
| 列 | `accel.csv`: `n,t_us,ax_g,ay_g,az_g` ／ `gps.csv`: `t_rx_us,utc,date,status,fix,sats,hdop,lat,lon,speed_kmh,course_deg` |
| 単位・周波数 | g（重力込み、±16 g）、実測 約 398.5 Hz |
| 時刻基準 | `t_us`（ESP32 起動からの us）。GPS 有効行から UTC へのオフセットを求めて JST を付ける。GPS なしは相対時間のみ |
| GPS 有効割合 | ride_004: 0.95、ride_005: 0.90、ride_006〜009: 0.00 |
| 欠け・エラー | なし（9 本とも） |
| 要注意 | ride_004 の 100 km/h 区間と ride_005 の 55 km/h 区間は低品質測位。ride_007 で X 軸 15.9 g が 1 回 |